# Quickstart — the core, with no IMP

Everything here runs on the pip package:

```bash
pip install --pre "imp-bff[notebooks]"
```

Run it from a clone of the repository, because it reads a structure from
`examples/structure/T4L/`. It takes about half a minute.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import IMP.bff as bff

print("build:", bff.get_build(), "· version:", bff.get_module_version())

## A dye's accessible volume

Where a dye on a flexible linker can be, given the structure it is attached to.
`get_av_from_pdb` takes the attachment site (chain, residue, atom) and the
linker's geometry, and returns the cloud, its density and the grid.

In [ ]:
pdb = "../../examples/structure/T4L/3GUN.pdb"   # T4 lysozyme

donor = bff.get_av_from_pdb(pdb, chain="A", resseq=132, atom_name="CB",
                            linker_length=20.0, linker_width=2.0, r1=3.5)
acceptor = bff.get_av_from_pdb(pdb, "A", 55, "CB", 20.0, 2.0, 3.5)

print(f"donor:    {donor.get_n_points():5d} points, mean position "
      f"{np.round(donor.get_mean_position(), 1)}")
print(f"acceptor: {acceptor.get_n_points():5d} points, mean position "
      f"{np.round(acceptor.get_mean_position(), 1)}")

## What the experiment would measure

The two clouds give a distance *distribution*, not one distance — which is why
a measured FRET efficiency does not convert into a single number by hand.
`mean_fret_distance` is the separation that reproduces the measured efficiency;
`average_distance` is the mean of the distribution. They differ.

In [ ]:
axis = np.linspace(10.0, 90.0, 81)
p = np.asarray(bff.cloud_distance_distribution(donor.get_points(),
                                               acceptor.get_points(), axis))

r_da = bff.average_distance(donor.get_points(), acceptor.get_points())
r_e = bff.mean_fret_distance(donor.get_points(), acceptor.get_points(), 52.0)
print(f"<R_DA> = {r_da:.1f} A      R_E(R0=52 A) = {r_e:.1f} A")

plt.figure(figsize=(5, 3))
plt.plot(axis[:len(p)], p)
plt.axvline(r_da, color="k", ls="--", lw=1, label=f"<R_DA> = {r_da:.1f} A")
plt.axvline(r_e, color="C3", ls=":", lw=1, label=f"R_E = {r_e:.1f} A")
plt.xlabel("R_DA (A)"); plt.ylabel("p(R_DA)"); plt.legend(); plt.tight_layout()

## A fluorescence decay

`TCSPCDecay` is the instrument model: a lifetime spectrum convolved with the
measured response, on the data's own time axis. It is a graph node — the same
one a fit optimises — so the amplitudes and lifetimes are ports you set.

In [ ]:
n, dt, period = 512, 0.032, 1000.0 / 80.0     # 512 channels, 32 ps, 80 MHz
time = np.arange(n) * dt
irf = 1000.0 * np.exp(-0.5 * ((time - 1.0) / 0.08) ** 2)

decay = bff.TCSPCDecay("decay")
decay.set_number_of_lifetimes(2)
decay.add_output_port("decay", bff.GraphPort([0.0], False, True))
decay.set_response_array(np.ascontiguousarray(irf))
decay.set_timing(dt, period)
decay.set_convolution_range(n, n)

for i, (amplitude, lifetime) in enumerate([(0.7, 4.0), (0.3, 1.2)]):
    decay.get_input_port(f"a{i}").value = amplitude
    decay.get_input_port(f"t{i}").value = lifetime

decay.update()
y = np.asarray(decay.get_output_port("decay").value)

plt.figure(figsize=(5, 3))
plt.semilogy(time, irf / irf.max(), color="0.6", label="response")
plt.semilogy(time, y / y.max(), label="decay (4.0 ns / 1.2 ns)")
plt.xlabel("time (ns)"); plt.ylabel("normalised counts")
plt.ylim(1e-4, 2); plt.legend(); plt.tight_layout()

## Where to go next

- `doc/manual/structure/structure_label_sites.ipynb` — scoring the labelling
  sites of a structure, also core-only.
- `examples/labels/plot_labelizer_score.py` — the same score as a script.
- The rest of `doc/manual/structure/` and most of `examples/labels/` build IMP
  restraints and need the conda module (`conda install -c conda-forge imp.bff`).
- `IMP.bff.registry()` lists the samplers, graph nodes and model families this
  build publishes.